# Databricks notebook source
# MAGIC %md
# MAGIC # 02 Silver Transformations
# MAGIC
# MAGIC This notebook reads the Bronze activity summary table, applies cleaning rules, and writes a Silver Delta table.


In [0]:
# COMMAND ----------

from pyspark.sql.functions import col, current_timestamp

# COMMAND ----------

In [0]:
# COMMAND ----------

bronze_table_name = "athlete_training_lakehouse.bronze_activity_summary"
silver_table_name = "athlete_training_lakehouse.silver_activity_summary"


In [0]:
# COMMAND ----------

bronze_df = spark.table(bronze_table_name)

display(bronze_df)

In [0]:
# COMMAND ----------

silver_df = (
    bronze_df
        .dropDuplicates(["activity_id"])
        .filter(col("distance_miles").isNotNull())
        .filter(col("distance_miles") > 0)
        .withColumn("silver_updated_at", current_timestamp())
)

In [0]:
# COMMAND ----------

(
    silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table_name)
)


In [0]:
# COMMAND ----------

display(spark.table(silver_table_name))

In [0]:
# COMMAND ----------

row_count = spark.table(silver_table_name).count()

print(f"Silver table created successfully: {silver_table_name}")
print(f"Rows loaded: {row_count}")

In [0]:
print(f"Bronze rows : {bronze_df.count()}")
print(f"Silver rows : {silver_df.count()}")

print(f"Duplicates removed: {bronze_df.count() - silver_df.count()}")